# DC-Ops — Train & Compile ExecuTorch models on Colab

Train / export the DC-Ops detection model, compile it to an ExecuTorch `.pte`, and download it to drop into the Android app (`android-app/app/src/main/assets/`).

**Two compile targets** — pick based on what you need:

| Path | Backend | Runs on | Setup cost | Reliability |
|------|---------|---------|-----------|-------------|
| **A** | **XNNPACK** | phone **CPU** | ~2 min (pip) | High — works on any Colab |
| **B** | **QNN / QAIRT** | Snapdragon **NPU** (Hexagon v79) | ~20-30 min (SDK + source build) | Lower — heavier, NPU-specific |

Start with **Path A** to get a working `.pte` immediately. Use **Path B** for the on-NPU performance the project targets (SM8750).

> ⚠️ **Status: scaffold.** This notebook encodes the working steps but has **not been executed end-to-end on Colab**. Run cell-by-cell and expect to iterate on the model-export cells (YOLO's detection head / NMS often need adjustment to lower cleanly). The toolchain-setup cells mirror a verified local build.

**Tip:** `Runtime → Change runtime type → GPU` accelerates *training only* — the ExecuTorch compile/lowering is CPU-bound.

In [ ]:
#@title Runtime check (OS / CPU / GPU)
import platform, subprocess, sys
print('Python :', sys.version.split()[0])
print('OS     :', platform.platform())
print('Arch   :', platform.machine(), '(QNN AOT needs x86_64-linux)')
try:
    print(subprocess.check_output(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader']).decode().strip(), '(GPU - training only)')
except Exception:
    print('No GPU (fine — compile is CPU-bound; enable GPU only to speed up training)')

---
## (Optional) Train / fine-tune YOLOv8n

There is **no dataset in the repo**, so this defaults to the pretrained `yolov8n.pt`. To train on the 6 DC-Ops classes (`led_green, led_amber, led_red, led_off, cable, label`), provide a YOLO-format dataset (e.g. a Roboflow export) and set `DATA_YAML`.

In [ ]:
#@title Load (and optionally train) YOLOv8n
!pip -q install ultralytics==8.3.0
from ultralytics import YOLO

DATA_YAML = ''   #@param {type:'string'}  # path/URL to a YOLO data.yaml; leave blank to skip training
EPOCHS    = 50   #@param {type:'integer'}
IMG_SIZE  = 640  #@param {type:'integer'}

model = YOLO('yolov8n.pt')  # pretrained COCO weights
if DATA_YAML.strip():
    print('Training on', DATA_YAML)
    model.train(data=DATA_YAML, epochs=EPOCHS, imgsz=IMG_SIZE)  # uses GPU if available
    model = YOLO('runs/detect/train/weights/best.pt')
    print('Trained weights: runs/detect/train/weights/best.pt')
else:
    print('No DATA_YAML -> using pretrained yolov8n.pt (good enough to validate the export pipeline)')

torch_model = model.model.eval()  # underlying nn.Module

---
# Path A — XNNPACK (phone CPU)  ✅ reliable, ~2 min

No QNN SDK or source build. Produces a quantized (INT8) `.pte` that the app loads via the ExecuTorch AAR. This is the fastest way to get *a* working model into the app.

In [ ]:
#@title A1. Install ExecuTorch (CPU wheel)
!pip -q install executorch
import executorch, torch
print('executorch', getattr(executorch, '__version__', '?'), '| torch', torch.__version__)

In [ ]:
#@title A2. Export YOLOv8n -> XNNPACK INT8 .pte
# NOTE: YOLO's full forward returns post-processed detections; some ops may not
# lower. If export fails, the standard fix is to export the raw forward and run
# NMS in Kotlin. See the troubleshooting cell at the bottom.
import torch
from torch.export import export
from executorch.exir import to_edge_transform_and_lower
from executorch.backends.xnnpack.partition.xnnpack_partitioner import XnnpackPartitioner
from executorch.backends.xnnpack.quantizer.xnnpack_quantizer import (
    XNNPACKQuantizer, get_symmetric_quantization_config,
)
from torchao.quantization.pt2e.quantize_pt2e import prepare_pt2e, convert_pt2e

IMG = 640
example = (torch.randn(1, 3, IMG, IMG),)

# 1) capture
ep = export(torch_model, example)
# 2) PT2E INT8 quantization (calibrate on a few random/representative inputs)
q = XNNPACKQuantizer().set_global(get_symmetric_quantization_config(is_per_channel=True))
m = prepare_pt2e(ep.module(), q)
for _ in range(8):
    m(torch.randn(1, 3, IMG, IMG))   # TODO: replace with real calibration images for accuracy
m = convert_pt2e(m)
# 3) lower to XNNPACK + serialize
ep_q = export(m, example)
prog = to_edge_transform_and_lower(ep_q, partitioner=[XnnpackPartitioner()]).to_executorch()

import os; os.makedirs('out', exist_ok=True)
OUT_XNN = 'out/yolov8n_int8_xnnpack.pte'
with open(OUT_XNN, 'wb') as f:
    f.write(prog.buffer)
print('Wrote', OUT_XNN, f'({os.path.getsize(OUT_XNN)/1e6:.1f} MB)')

In [ ]:
#@title A3. ⬇️ Download the .pte (and back up to Drive)
from google.colab import files
files.download(OUT_XNN)
# Optional: also save to Drive so a Colab disconnect doesn't lose it
try:
    from google.colab import drive; drive.mount('/content/drive')
    import shutil; shutil.copy(OUT_XNN, '/content/drive/MyDrive/'); print('Backed up to Drive')
except Exception as e:
    print('Drive backup skipped:', e)

---
# Path B — QNN / QAIRT (Snapdragon NPU)  ⚙️ heavy, ~20-30 min

Builds ExecuTorch **with the Qualcomm QNN backend** and compiles for **SM8750 (Hexagon v79)** — the NPU path the project targets. Slower and more fragile than Path A. Colab is `x86_64-linux` and you are **root**, so the dependency fixes are clean (no sudo/PPA pain).

Cells: B1 system deps → B2 QNN SDK + NDK → B3 build ExecuTorch QNN → B4 smoke-test toolchain → B5 export YOLO → B6 download.

In [ ]:
#@title B1. System deps (g++13, cmake<4, libc++ / libunwind trio)
# These are the exact blockers hit during the local build. On Colab (root) apt resolves them cleanly.
!sudo apt-get -qq update
!sudo apt-get -qq install -y g++-13 libc++1 libc++abi1 libunwind8 || true
# QNN's clang-built x86 libs want the LLVM soname libunwind.so.1; apt ships .so.8 -> symlink it.
!for d in /usr/lib/x86_64-linux-gnu; do [ -e $d/libunwind.so.8 ] && ln -sf $d/libunwind.so.8 $d/libunwind.so.1; done
!pip -q install 'cmake<4' ninja
import os
os.environ['CC'] = 'gcc-13'; os.environ['CXX'] = 'g++-13'
!g++-13 --version | head -1 && cmake --version | head -1

In [ ]:
#@title B2. Download QAIRT (QNN) SDK 2.46.0.260424 + Android NDK r26c
import os
QNN_URL = 'https://softwarecenter.qualcomm.com/api/download/software/sdks/Qualcomm_AI_Runtime_Community/All/2.46.0.260424/v2.46.0.260424.zip'
QNN_SHA = '2db536eac7556368f4e89663bd26aa0af189cf16360d74ae48324a5281f9a4e1'
NDK_URL = 'https://dl.google.com/android/repository/android-ndk-r26c-linux.zip'
# HEAD 403s on Qualcomm's gateway but GET works; force HTTP/1.1 + resume to avoid stream errors.
!cd /content && curl -L --http1.1 -C - --retry 8 --retry-all-errors -o qnn.zip "$QNN_URL"
!cd /content && echo "$QNN_SHA  qnn.zip" | sha256sum -c - || echo 'WARNING: SHA mismatch (re-run to resume the download)'
!cd /content && unzip -q -o qnn.zip && unzip -q -o <(curl -Ls "$NDK_URL") -d ndk 2>/dev/null || (curl -L -o ndk.zip "$NDK_URL" && unzip -q -o ndk.zip -d ndk)
os.environ['QNN_SDK_ROOT']     = '/content/qairt/2.46.0.260424'
os.environ['ANDROID_NDK_ROOT'] = '/content/ndk/android-ndk-r26c'
print('QNN_SDK_ROOT     =', os.environ['QNN_SDK_ROOT'], os.path.isdir(os.environ['QNN_SDK_ROOT']))
print('ANDROID_NDK_ROOT =', os.environ['ANDROID_NDK_ROOT'], os.path.isdir(os.environ['ANDROID_NDK_ROOT']))

In [ ]:
#@title B3. Build ExecuTorch with the Qualcomm backend (~20-30 min)
import os
if not os.path.isdir('/content/executorch'):
    !cd /content && git clone https://github.com/pytorch/executorch.git
    !cd /content/executorch && git submodule sync && git submodule update --init --recursive
os.environ['EXECUTORCH_ROOT'] = '/content/executorch'
os.environ['PYTHONPATH'] = '/content:' + os.environ.get('PYTHONPATH','')
os.environ['LD_LIBRARY_PATH'] = f"{os.environ['QNN_SDK_ROOT']}/lib/x86_64-linux-clang:" + os.environ.get('LD_LIBRARY_PATH','')
!cd /content/executorch && ./install_executorch.sh
!cd /content/executorch && source $QNN_SDK_ROOT/bin/envsetup.sh && ./backends/qualcomm/scripts/build.sh
print('Build done. PyQnnManagerAdaptor present:',
      bool(__import__('glob').glob('/content/executorch/build-x86/**/PyQnnManagerAdaptor*.so', recursive=True)))

In [ ]:
#@title B4. Smoke-test the toolchain (mobilenet_v2 -> .pte must be non-empty)
# Proves QNN AOT works BEFORE risking the YOLO-specific export.
import os, glob
os.environ['QNN_SDK_ROOT'] = '/content/qairt/2.46.0.260424'
code = '''
import torch
from executorch.backends.qualcomm.export_utils import build_executorch_binary, QnnConfig
from executorch.backends.qualcomm.quantizer.quantizer import QuantDtype
from executorch.examples.models.mobilenet_v2 import MV2Model
cfg = QnnConfig(soc_model="SM8750", backend="htp")
build_executorch_binary(model=MV2Model().get_eager_model().eval(), qnn_config=cfg,
    file_name="/content/out/mv2_qnn", dataset=[(torch.rand(1,3,224,224),)],
    quant_dtype=QuantDtype.use_8a8w)
'''
os.makedirs('/content/out', exist_ok=True)
open('/content/_smoke.py','w').write(code)
!cd /content/executorch && PYTHONPATH=/content:$PYTHONPATH python /content/_smoke.py
p = '/content/out/mv2_qnn.pte'
print('SMOKE OK' if os.path.exists(p) and os.path.getsize(p) > 0 else 'SMOKE FAILED', '->', p)

In [ ]:
#@title B5. Export YOLOv8n -> QNN INT8 .pte (SM8750)
# Reuses `torch_model` from the training cell. If lowering fails on detection-head
# ops, pass skip nodes via QnnConfig or export the raw backbone forward + NMS-in-Kotlin.
import os, torch
from executorch.backends.qualcomm.export_utils import build_executorch_binary, QnnConfig
from executorch.backends.qualcomm.quantizer.quantizer import QuantDtype

IMG = 640
calib = [(torch.randn(1,3,IMG,IMG),) for _ in range(8)]  # TODO: real images for accuracy
cfg = QnnConfig(soc_model='SM8750', backend='htp')
OUT_QNN = '/content/out/yolov8n_int8'
build_executorch_binary(model=torch_model, qnn_config=cfg, file_name=OUT_QNN,
                        dataset=calib, quant_dtype=QuantDtype.use_8a8w)
print('Wrote', OUT_QNN + '.pte', f'({os.path.getsize(OUT_QNN+".pte")/1e6:.1f} MB)')

In [ ]:
#@title B6. ⬇️ Download the QNN .pte (and back up to Drive)
from google.colab import files
files.download('/content/out/yolov8n_int8.pte')
try:
    from google.colab import drive; drive.mount('/content/drive')
    import shutil; shutil.copy('/content/out/yolov8n_int8.pte', '/content/drive/MyDrive/'); print('Backed up to Drive')
except Exception as e:
    print('Drive backup skipped:', e)

---
## Putting the `.pte` into the DC-Ops app

The download gives you the `.pte`. To actually use it, three things are still required in the app (currently the ML is stubbed):

1. **Drop the file in assets:** copy the downloaded `.pte` to `android-app/app/src/main/assets/yolov8n_int8.pte` (create the `assets/` folder if missing).
2. **Add the ExecuTorch AAR** to `android-app/app/build.gradle.kts` (`implementation(files("libs/executorch.aar"))`, built via `executorch/scripts/build_android_library.sh`). For the **QNN** `.pte` the AAR must include the QNN backend + the v79 runtime libs are packaged.
3. **Un-stub `ModelManager.kt`:** replace the sample-polygon body with `Module.load(assetFilePath(ctx, "yolov8n_int8.pte"))`, convert each `ImageProxy` to an NCHW float tensor (resize 640×640, normalize), run `module.forward(...)`, and parse outputs (+ NMS) into `DetectionResult`s.

Rebuild & install: `gradlew :app:assembleDebug` → `adb install -r app-debug.apk`.

### Troubleshooting export
- **Op not supported / lowering fails:** YOLO's detection head + NMS are the usual culprits. Export the model's raw forward (regression+cls outputs) and do decoding/NMS in Kotlin, or pass `skip_delegate_node_ops` in `QnnConfig` to fall back those ops to CPU.
- **`pip install executorch` lacks QNN AOT:** the pip wheel covers Path A (XNNPACK); Path B needs the source build (B3) — that's what produces `PyQnnManagerAdaptor.so`.
- **QNN lib load error (`libc++.so.1` / `libunwind.so.1`):** re-run B1 (the apt installs + libunwind symlink).